# Prepare Annotations for InferCNV

##### Franziska Niemeyer, 2026-05-23

In [ ]:
import scanpy as sc
import squidpy as sq
import pandas as pd
import anndata as ad
import matplotlib.pyplot as plt
import matplotlib.image as mpimg
import seaborn as sns
import anndata as ad
import numpy as np
import os
import cmcrameri.cm as cmc

from pathlib import Path
from aquarel import load_theme
from scipy.sparse import issparse

In [ ]:
WORKING_DIR = "../.."
OUT_DIR = f"{WORKING_DIR}/cnv_inference"
ADATA = "../../../../quality_control/external-cohort/pre-processing/adata.h5ad"

adata = ad.read_h5ad(ADATA)

In [ ]:
adata

In [ ]:
sample_key = "sample"
annotation_key = "histology"

# keep only spots with a non-missing annotation
adata = adata[
    (adata.obs[annotation_key].notna())
].copy()

print(adata)

In [ ]:
# Exclude necrotic debris, Blood (intravascular), and Ovarian surface epithelium
adata = adata[
    ~adata.obs[annotation_key].isin(
        ["Necrotic debris", "Blood (intravascular)", "Blood- hemorrhage", "Ovarian surface epithelium", "Tumor epithelium and stroma", "Tumor epithelium and ovarian stroma"]
    )
].copy()

In [ ]:
adata.obs[annotation_key].value_counts()

In [ ]:
adata.obs['annotation'] = adata.obs['sample'].astype(str) + " - " + adata.obs['patient'].astype(str) + " - " + adata.obs['histology'].astype(str)

In [ ]:
adata.obs['annotation'].value_counts()

In [ ]:
# Count annotations with less than ten barcodes
print(adata.obs['annotation'].value_counts()[adata.obs['annotation'].value_counts() < 10])

In [ ]:
# Remove annotations with less than 10 barcodes
adata = adata[adata.obs['annotation'].map(adata.obs['annotation'].value_counts()) >= 10].copy()

In [ ]:
# keep the current gene symbols
adata.var["gene_symbol"] = adata.var_names.astype(str)

# switch var_names to Ensembl IDs
adata.var_names = adata.var["gene_ids"].astype(str)

print(adata.var_names.is_unique)
print(adata.var_names[:5])

In [ ]:
out_dir = os.path.join(OUT_DIR, "infercnv_input")
os.makedirs(out_dir, exist_ok=True)

filtered_h5ad_path = os.path.join(out_dir, "visium_filtered_for_infercnv.h5ad")
adata.write(filtered_h5ad_path)

print(f"Saved filtered AnnData to: {filtered_h5ad_path}")

In [ ]:
from pathlib import Path
import gzip
import numpy as np
import pandas as pd
import scipy.sparse as sp
import h5py
from scipy.io import mmwrite


def export_infercnv_inputs_combined(
    adata,
    out_dir,
    sample_key="slide",
    annotation_key="annotation",
    patient_key="sample",
    reference_annotation="Stroma",
    layer=None,
    prefix="combined",
    filter_patients=None,
    use_full_obs_names=False,
    obs_name_sep="_",
    write_mtx=True,
    write_tsv=False,
    write_h5=True,
):
    """
    Export combined inferCNV input files from an AnnData object.

    Observation / reference split
    ------------------------------
    filter_patients : list[str] | None
        When provided, only spots belonging to these patients are exported as
        observations (non-reference annotations).  The reference set is always
        built from ALL patients in the dataset (any spot whose annotation ==
        reference_annotation), so the reference pool is never restricted.
        When None, all patients are used as observations.

    Annotation handling
    -------------------
    Non-reference spots  ->  "{patient_id} - {annotation}"
    Reference spots      ->  "{reference_annotation}"   (pooled, all patients)

    Notes
    -----
    - inferCNV expects raw counts, not log-normalized values.
    - The annotation file row order matches the count matrix columns.
    """

    out_dir = Path(out_dir)
    out_dir.mkdir(parents=True, exist_ok=True)

    # ── Choose layer ──────────────────────────────────────────────────────
    X_full = adata.layers[layer] if layer is not None else adata.X
    if not sp.issparse(X_full):
        X_full = sp.csr_matrix(np.asarray(X_full))
    X_full = X_full.tocsr()

    raw_annotations = adata.obs[annotation_key].astype(str).values
    patient_ids     = adata.obs[patient_key].astype(str).values

    # ── Build index masks ─────────────────────────────────────────────────
    is_reference = raw_annotations == reference_annotation

    if filter_patients is not None:
        filter_patients_set = set(str(p) for p in filter_patients)
        is_target_patient   = np.isin(patient_ids, list(filter_patients_set))
    else:
        is_target_patient   = np.ones(adata.n_obs, dtype=bool)

    # Observations: target-patient spots that are NOT the reference annotation
    obs_mask = is_target_patient & ~is_reference
    # Reference: ALL reference spots across the entire dataset
    ref_mask = is_reference

    if obs_mask.sum() == 0:
        raise ValueError(
            f"No non-reference spots found for patient(s) {filter_patients}. "
            "Check filter_patients and annotation_key."
        )
    if ref_mask.sum() == 0:
        raise ValueError(
            f"No reference spots found with annotation '{reference_annotation}'."
        )

    # Combined index: observations first, then reference
    combined_mask    = obs_mask | ref_mask
    combined_indices = np.where(combined_mask)[0]

    # Keep obs-then-ref ordering explicitly
    obs_indices = np.where(obs_mask)[0]
    ref_indices = np.where(ref_mask)[0]
    ordered_indices = np.concatenate([obs_indices, ref_indices])

    # ── Count matrix (genes × spots) ─────────────────────────────────────
    X_sub  = X_full[ordered_indices, :]          # spots × genes
    X_gc   = X_sub.T.tocsr()                     # genes × spots

    # ── Barcodes ─────────────────────────────────────────────────────────
    sub_obs        = adata.obs.iloc[ordered_indices]
    sub_obs_names  = adata.obs_names[ordered_indices].astype(str)

    if use_full_obs_names:
        barcodes = np.array(sub_obs_names)
    else:
        sample_names     = sub_obs[sample_key].astype(str).values
        stripped_barcodes = []
        for obs_name, sample in zip(sub_obs_names, sample_names):
            obs_prefix = f"{sample}{obs_name_sep}"
            stripped_barcodes.append(
                obs_name[len(obs_prefix):] if obs_name.startswith(obs_prefix) else obs_name
            )
        stripped_barcodes = np.array(stripped_barcodes)

        if pd.Index(stripped_barcodes).duplicated().any():
            print(
                "[warn] Stripped barcodes are not unique across samples. "
                "Using full obs_names instead."
            )
            barcodes = np.array(sub_obs_names)
        else:
            barcodes = stripped_barcodes

    # ── Genes ─────────────────────────────────────────────────────────────
    genes = np.array(adata.var_names.astype(str))

    # ── Annotations ───────────────────────────────────────────────────────
    sub_raw_annotations = raw_annotations[ordered_indices]
    sub_patient_ids     = patient_ids[ordered_indices]

    export_annotations = np.where(
        sub_raw_annotations == reference_annotation,
        reference_annotation,
        sub_patient_ids + " - " + sub_raw_annotations,
    )

    # ── Diagnostics ───────────────────────────────────────────────────────
    print(f"Spots in output : {len(ordered_indices)}  "
          f"({obs_mask.sum()} observations + {ref_mask.sum()} reference)")
    print(f"Genes           : {adata.n_vars}  |  example: {genes[:3]}")
    if filter_patients is not None:
        print(f"Observation patients (filter_patients): {sorted(filter_patients_set)}")
    print(f"Reference annotation '{reference_annotation}': {ref_mask.sum()} spots "
          f"(from all {len(set(patient_ids[ref_mask]))} patients)")
    print(f"Exported annotation value counts:\n{pd.Series(export_annotations).value_counts()}")

    # ── Annotation file ───────────────────────────────────────────────────
    ann_df   = pd.DataFrame({"barcode": barcodes, "annotation": export_annotations})
    ann_path = out_dir / f"{prefix}_infercnv_annotations.tsv"
    ann_df.to_csv(ann_path, sep="\t", header=False, index=False)

    # ── Barcodes / genes ──────────────────────────────────────────────────
    with gzip.open(out_dir / f"{prefix}_barcodes.tsv.gz", "wt") as f:
        f.writelines(f"{b}\n" for b in barcodes)

    with gzip.open(out_dir / f"{prefix}_genes.tsv.gz", "wt") as f:
        f.writelines(f"{g}\n" for g in genes)

    # ── Matrix Market ─────────────────────────────────────────────────────
    if write_mtx:
        mtx_path = out_dir / f"{prefix}_counts.mtx"
        mmwrite(str(mtx_path), X_gc)
        with open(mtx_path, "rb") as f_in, \
             gzip.open(f"{mtx_path}.gz", "wb") as f_out:
            f_out.writelines(f_in)
        mtx_path.unlink()

    # ── Dense TSV ─────────────────────────────────────────────────────────
    if write_tsv:
        dense    = X_gc.toarray() if sp.issparse(X_gc) else np.asarray(X_gc)
        df_counts = pd.DataFrame(dense, index=genes, columns=barcodes)
        df_counts.to_csv(out_dir / f"{prefix}_counts.tsv.gz",
                         sep="\t", compression="gzip")

    # ── Custom HDF5 ───────────────────────────────────────────────────────
    if write_h5:
        counts = X_gc.toarray() if sp.issparse(X_gc) else np.asarray(X_gc)
        with h5py.File(out_dir / f"{prefix}_counts.h5", "w") as h5:
            h5.create_dataset("counts",      data=counts,                    compression="gzip")
            h5.create_dataset("genes",       data=genes.astype("S"))
            h5.create_dataset("barcodes",    data=barcodes.astype("S"))
            h5.create_dataset("annotations", data=export_annotations.astype("S"))

    print(f"Wrote inferCNV input files to: {out_dir}")

In [ ]:
export_infercnv_inputs_combined(
    adata=adata,
    out_dir=os.path.join(out_dir, "tumor-stroma-ova09"),
    sample_key="sample",
    annotation_key="histology",
    layer="counts",
    prefix="tumor-stroma-ova09",
    patient_key="patient",
    filter_patients=["OVA09"],
    reference_annotation="Ovarian stroma",
    use_full_obs_names=True,
    write_mtx=True,
    write_tsv=False,
    write_h5=True,
)